# 08 Static Maps and Report Figures

This notebook creates simple static maps and charts for reports, portfolio documentation, and README visuals.

## What you will do

1. Check which processed rasters and vectors are available.
2. Generate raster preview maps.
3. Generate vector/overlay preview maps.
4. Create simple summary charts.
5. Identify outputs that still need polished cartography.

## Step 1: Load configuration and inspect available outputs

In [ ]:
from pathlib import Path
import pandas as pd
from floodsense.config import load_config
from floodsense.paths import list_input_files

config = load_config()
paths = config['paths']

print('Processed rasters:')
for file in list_input_files(paths['processed_rasters'], ['.tif', '.tiff']):
    print('  -', file.name)

print('\nProcessed vectors:')
for file in list_input_files(paths['processed_vectors'], ['.gpkg', '.geojson']):
    print('  -', file.name)

## Step 2: Generate raster previews

These previews are quick checks. For a final portfolio map, you may later add legends, scale bars, north arrows, labels, and consistent styling.

In [ ]:
from floodsense.mapping import plot_raster_preview

maps = Path(paths['outputs_maps'])
raster_previews = {
    'susceptibility_class.tif': 'Flood Susceptibility Classes',
    'susceptibility_score.tif': 'Flood Susceptibility Score',
    'observed_flood_extent.tif': 'SAR Observed Flood Extent',
    'model_vs_observed_comparison.tif': 'Model vs Observed Flood Comparison',
}

for filename, title in raster_previews.items():
    raster = Path(paths['processed_rasters']) / filename
    if raster.exists():
        output = maps / f"{Path(filename).stem}_report_preview.png"
        plot_raster_preview(raster, output, title)
        print('Saved:', output)
    else:
        print('Missing:', filename)

## Step 3: Generate vector and overlay previews

In [ ]:
from floodsense.mapping import plot_vector_preview, plot_overlay_preview

vectors = Path(paths['processed_vectors'])
boundary = vectors / 'lokoja_boundary.gpkg'
flood_zones = vectors / 'high_risk_flood_zones.gpkg'
buildings = vectors / 'exposed_buildings.gpkg'
roads = vectors / 'exposed_roads.gpkg'

if flood_zones.exists():
    plot_vector_preview(flood_zones, maps / 'high_risk_flood_zones_report_preview.png', 'High-Risk Flood Zones')
if boundary.exists() and flood_zones.exists():
    plot_overlay_preview(boundary, flood_zones, maps / 'boundary_flood_zones_overlay.png', 'Lokoja Boundary and Flood Zones')
if boundary.exists() and buildings.exists():
    plot_overlay_preview(boundary, buildings, maps / 'exposed_buildings_overlay.png', 'Exposed Buildings')
if boundary.exists() and roads.exists():
    plot_overlay_preview(boundary, roads, maps / 'exposed_roads_overlay.png', 'Exposed Roads')

## Step 4: Create summary charts from tables

In [ ]:
from floodsense.mapping import save_bar_chart

class_area = Path(paths['processed_tables']) / 'susceptibility_class_area.csv'
if class_area.exists():
    df = pd.read_csv(class_area)
    save_bar_chart(df, 'class_name', 'area_km2', maps / 'susceptibility_class_area_chart.png', 'Area by Flood Susceptibility Class')
    display(df)
else:
    print('susceptibility_class_area.csv is missing.')

priority = Path(paths['processed_tables']) / 'priority_ranking.csv'
if priority.exists():
    priority_df = pd.read_csv(priority)
    if 'priority_class' in priority_df.columns:
        counts = priority_df['priority_class'].value_counts().rename_axis('priority_class').reset_index(name='count')
        save_bar_chart(counts, 'priority_class', 'count', maps / 'priority_class_count_chart.png', 'Priority Class Counts')
        display(counts)
else:
    print('priority_ranking.csv is missing.')

## What to check before moving on

- Are the preview maps visually reasonable?
- Which figures should go into the README or portfolio case study?
- Which maps need more careful cartographic styling?

Next notebook: `09_dashboard_layer_preparation.ipynb`.